# Clase 176 — Test t: una muestra, dos muestras, pareado

El p-value responde *¿hay diferencia?*; el effect size responde *¿cuánta?*. Recorremos las tres variantes del test t con `scipy.stats`, leemos p-value e intervalo de confianza, usamos **Welch** por default y reportamos **Cohen's d**.

Requiere: `numpy`, `scipy`, `matplotlib`.

## 1. Test t de una muestra

`H₀: μ = 20` vs `H₁: μ ≠ 20`. El IC95 % y el p-value son la misma información: si el IC excluye 20, se rechaza `H₀`.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

total_bill = rng.normal(21.5, 8, 200)
res = stats.ttest_1samp(total_bill, popmean=20)
ci = res.confidence_interval(0.95)
print("H0: μ=20  vs  H1: μ≠20")
print(f"t={res.statistic:.3f}  p={res.pvalue:.4f}  IC95%=({ci.low:.2f}, {ci.high:.2f})")
print("Rechazo H0" if res.pvalue < 0.05 else "No rechazo H0")
excluye_20 = ci.low > 20 or ci.high < 20
assert (res.pvalue < 0.05) == excluye_20, "p-value e IC deben concordar" 

## 2. Dos muestras independientes (Welch) + Cohen's d

Welch no asume varianzas iguales: es el default razonable moderno. Reportamos siempre effect size.

In [ ]:
tip_a = rng.normal(3.0, 1.1, 120)
tip_b = rng.normal(3.5, 1.3, 100)
res = stats.ttest_ind(tip_a, tip_b, equal_var=False)   # Welch
print(f"Welch t={res.statistic:.3f}  p={res.pvalue:.4f}")

def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

d = cohens_d(tip_a, tip_b)
print(f"Cohen's d = {d:.3f}  (|d|: 0.2 small, 0.5 medium, 0.8 large)")
assert res.pvalue < 0.05

## 3. Test t pareado

Cada sujeto es su propio control (presión arterial antes/después). El pareado elimina la varianza entre sujetos y gana muchísimo poder frente al `ttest_ind` mal aplicado.

In [ ]:
antes = rng.normal(140, 12, 30)
despues = antes - rng.normal(5, 3, 30)     # baja ~5 mmHg
rel = stats.ttest_rel(antes, despues)
ind = stats.ttest_ind(antes, despues)      # INCORRECTO: ignora el pareo
print(f"Pareado (ttest_rel): t={rel.statistic:.3f}  p={rel.pvalue:.2e}")
print(f"Ind. incorrecto:     t={ind.statistic:.3f}  p={ind.pvalue:.3f}")
assert rel.pvalue < ind.pvalue, "el pareado tiene más poder"

plt.figure(figsize=(6, 4))
plt.hist(antes - despues, bins=15, color="teal", alpha=0.7)
plt.axvline(0, color="k", ls="--")
plt.title("Diferencias antes-después (pareado)"); plt.xlabel("mmHg")
plt.tight_layout(); plt.show()

## 4. Bilateral vs unilateral

La dirección debe fijarse **antes** de ver los datos (si no, es p-hacking). Cuando la dirección acierta, el p unilateral es ~la mitad del bilateral.

In [ ]:
res_two = stats.ttest_ind(tip_b, tip_a, equal_var=False, alternative="two-sided")
res_gr  = stats.ttest_ind(tip_b, tip_a, equal_var=False, alternative="greater")
print(f"bilateral            p={res_two.pvalue:.4f}")
print(f"unilateral (greater) p={res_gr.pvalue:.4f}")
print(f"ratio ≈ {res_two.pvalue / res_gr.pvalue:.2f}")
assert abs(res_gr.pvalue - res_two.pvalue / 2) < 1e-6

## 5. Significativo ≠ relevante

Con `n` enorme, cualquier diferencia trivial es "significativa". Por eso hay que reportar effect size + IC, no solo el p-value.

In [ ]:
grupo_a = rng.normal(100, 15, 10_000)
grupo_b = rng.normal(100.5, 15, 10_000)
res = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
d_big = cohens_d(grupo_b, grupo_a)
print(f"n=10000/grupo: p={res.pvalue:.4f}   Cohen's d = {d_big:.3f}")
print("p bajo pero d trivial: la diferencia NO es relevante en la práctica.")
assert abs(d_big) < 0.1

## Ejercicios

1. Repetí el test de una muestra con `popmean=21` y verificá cómo cambian t, p e IC.
2. Para el ejercicio 2, calculá también Hedges' g corregido `g = d·(1 - 3/(4(n₁+n₂)-9))` y compará con d.
3. En el caso pareado, aplicá `ttest_1samp(antes-despues, 0)` y confirmá que da exactamente lo mismo que `ttest_rel`.
4. Simulá dos grupos con `d≈0.5` y `n=64`; calculá el poder con `statsmodels.stats.power.TTestIndPower`.

## Conclusiones

- Usá **Welch** (`equal_var=False`) por default en dos muestras: no cuesta poder y protege ante varianzas distintas.
- El p-value y el IC95 % de la diferencia son la misma información; el IC es más informativo (magnitud + dirección).
- El test pareado elimina la varianza entre sujetos: mucho más poder cuando hay pareo natural.
- Reportá siempre effect size: con `n` grande todo es significativo pero no todo es relevante.